# Redes neurais e modelo fundacional de série

Os três notebooks anteriores foram de métodos que se ajustam a uma série só: ETS, SARIMAX e boosting com
atributos. Este trata dos dois extremos da prática moderna.

- **rede neural recorrente**: uma GRU que recebe uma janela dos últimos valores e aprende a prever o próximo.
  Ela não recebe nenhuma noção de tendência, sazonalidade ou ordem: aprende tudo do dado, e precisa de
  treino;
- **modelo fundacional de série**: um modelo já treinado em milhares de séries de domínios diferentes, que
  prevê **sem treinar nada** na sua. Basta entregar a janela de contexto.

A pergunta que organiza o notebook é sempre a mesma da pasta, e agora com os extremos dentro: **cada um
deles bate o sarrafo do sazonal ingênuo, e bate o ETS de quatro linhas?** E, no caso da rede, quanto custa
a resposta, em treino, em variação entre sementes e em número de decisões que alguém precisa tomar.

A série é a mesma dos três notebooks anteriores, com os mesmos componentes declarados.


In [ ]:
import copy
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from statsmodels.tsa.api import ExponentialSmoothing, SARIMAX

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (11, 4)})
warnings.filterwarnings("ignore")
torch.manual_seed(42)

RANDOM_STATE = 42
PERIODO = 7
JANELA = 28  # o tamanho do histórico que a rede e o modelo fundacional recebem


def serie_com_componentes(n_dias=760, seed=RANDOM_STATE):
    """A mesma série dos notebooks anteriores: tendência, semana, ano, ruído e um choque."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_dias)
    indice = pd.date_range("2024-01-01", periods=n_dias, freq="D")
    valores = (100 + 0.06 * t
               + 6.0 * np.sin(2 * np.pi * t / 7)
               + 12.0 * np.sin(2 * np.pi * t / 365)
               + rng.normal(0, 3.0, n_dias)
               + np.where((t >= 400) & (t < 415), 25.0 * np.exp(-(t - 400) / 4.0), 0.0))
    return pd.Series(valores, index=indice, name="vendas")


def erro_absoluto_medio(observado, previsto):
    return float(np.mean(np.abs(np.asarray(observado) - np.asarray(previsto))))


def mase(observado, previsto, treino, periodo=PERIODO):
    return erro_absoluto_medio(observado, previsto) / float(np.mean(np.abs(treino.diff(periodo).dropna())))


def backtest(serie, previsor, origens=9, passos=14):
    por_origem, erro_por_passo = [], []
    for origem in range(len(serie) - passos * origens, len(serie) - passos, passos):
        treino, alvo = serie.iloc[:origem], serie.iloc[origem:origem + passos]
        previsto = previsor(treino, passos)
        por_origem.append({"MAE": erro_absoluto_medio(alvo, previsto),
                           "MASE": mase(alvo, previsto, treino)})
        erro_por_passo.append(np.abs(alvo.to_numpy() - np.asarray(previsto)))
    return pd.DataFrame(por_origem), np.vstack(erro_por_passo)


serie = serie_com_componentes()
print(f"série: {len(serie)} dias | contexto de {JANELA} dias por previsão")
print("sarrafo: sazonal ingênuo com MAE médio 4,040 no backtesting")


## A rede neural: janela dentro, número fora

Uma rede recorrente em série temporal funciona assim: o histórico recente entra como janela, a rede devolve
o próximo valor, e a previsão de 14 passos sai do modo recursivo, exatamente como no boosting.

Três decisões são obrigatórias, e cada uma tem um jeito certo:

1. **escalonar o dado**. A rede aprende por gradiente, e uma série em torno de 100 com desvio 3 tem que
   chegar normalizada. Média e desvio vêm **do treino**, nunca do conjunto inteiro, senão a escala carrega
   informação do futuro;
2. **validar no fim do treino**. O pedaço de validação que decide quando parar é o fim do treino, em ordem,
   nunca um sorteio;
3. **escolher o ponto de parada**. Sem isso a rede decora o treino, e o erro de validação sobe enquanto o
   erro de treino desce.

## O que a rede entrega

A célula treina a GRU em cada origem do backtesting e mede o erro. É a mesma avaliação dos notebooks
anteriores, para o confronto ser honesto.


In [ ]:
class PrevisorGRU(nn.Module):
    """Uma GRU pequena: lê a janela e devolve um número."""

    def __init__(self, oculto=32):
        super().__init__()
        self.recorrente = nn.GRU(1, oculto, batch_first=True)
        self.saida = nn.Linear(oculto, 1)

    def forward(self, entrada):
        saida, _ = self.recorrente(entrada)
        return self.saida(saida[:, -1]).squeeze(-1)


def janelas(valores, tamanho=JANELA):
    entrada = np.stack([valores[i:i + tamanho] for i in range(len(valores) - tamanho)])
    return entrada, valores[tamanho:]


def treina_gru(treino, epocas=30, seed=RANDOM_STATE, oculto=32, fracao_validacao=0.15):
    """Treina na parte inicial do treino e para quando o fim dele começa a piorar."""
    torch.manual_seed(seed)
    valores = treino.to_numpy(dtype=np.float32)
    media, desvio = float(valores.mean()), float(valores.std())
    normalizado = (valores - media) / desvio

    corte = int(len(normalizado) * (1 - fracao_validacao))
    entrada, alvo = janelas(normalizado)
    fim_treino = corte - JANELA
    entrada_treino = torch.tensor(entrada[:fim_treino]).unsqueeze(-1)
    alvo_treino = torch.tensor(alvo[:fim_treino])
    entrada_validacao = torch.tensor(entrada[fim_treino:]).unsqueeze(-1)
    alvo_validacao = torch.tensor(alvo[fim_treino:])

    modelo = PrevisorGRU(oculto)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    perda = nn.MSELoss()

    melhor_perda, melhor_estado, historico = np.inf, None, []
    for _ in range(epocas):
        modelo.train()
        for inicio in range(0, len(entrada_treino), 64):
            lote_x = entrada_treino[inicio:inicio + 64]
            lote_y = alvo_treino[inicio:inicio + 64]
            otimizador.zero_grad()
            perda(modelo(lote_x), lote_y).backward()
            otimizador.step()
        modelo.eval()
        with torch.no_grad():
            erro_validacao = float(perda(modelo(entrada_validacao), alvo_validacao))
        historico.append(erro_validacao)
        if erro_validacao < melhor_perda:
            melhor_perda, melhor_estado = erro_validacao, copy.deepcopy(modelo.state_dict())

    modelo.load_state_dict(melhor_estado)
    return modelo, media, desvio, historico


def previsor_gru(treino, horizonte, seed=RANDOM_STATE):
    modelo, media, desvio, _ = treina_gru(treino, seed=seed)
    historico = (treino.to_numpy(dtype=np.float32) - media) / desvio
    modelo.eval()
    previsoes = []
    for _ in range(horizonte):
        with torch.no_grad():
            janela = torch.tensor(historico[-JANELA:], dtype=torch.float32).reshape(1, JANELA, 1)
            proximo = float(modelo(janela)) * desvio + media
        previsoes.append(proximo)
        historico = np.append(historico, (proximo - media) / desvio)
    return np.array(previsoes)


t0 = time.perf_counter()
_, passos_gru = backtest(serie, previsor_gru)
tempo_gru = time.perf_counter() - t0

print("GRU no backtesting:")
print(f"  MAE médio {passos_gru.mean():.3f} | passo 1 {passos_gru[:, 0].mean():.3f} | "
      f"passo 14 {passos_gru[:, -1].mean():.3f}")
print(f"  tempo: {tempo_gru:.1f} s em {passos_gru.shape[0]} origens "
      f"({tempo_gru / passos_gru.shape[0]:.1f} s por treino)")


A rede entregou 4,329 de MAE médio, e as outras duas colunas são o ponto: 2,632 no passo 1, melhor que todos
os métodos já treinados nesta pasta, e **7,237 no passo 14, o pior de todos**. A rede prevê muito bem um dia
à frente e desmonta ao longo do horizonte, porque é recursiva: cada previsão entra como dado para a próxima,
e o erro acumula. Custa menos de 3 segundos por treino, contra menos de um quarto de segundo de um ajuste de
ETS.

## A variação entre sementes

Rede neural não é determinística: a inicialização dos pesos é sorteada, e duas rodadas com os mesmos dados e
os mesmos hiperparâmetros dão notas diferentes. Isso é um custo que os métodos anteriores não têm, e vale
medir o tamanho dele: mesma origem, três sementes.


In [ ]:
origem = len(serie) - 14 * 7  # uma origem fixa, no meio do backtesting
treino_fixo = serie.iloc[:origem]
alvo_fixo = serie.iloc[origem:origem + 14]

linhas = []
for seed in (42, 7, 2026):
    t0 = time.perf_counter()
    previsto = previsor_gru(treino_fixo, 14, seed=seed)
    linhas.append({"semente": seed, "MAE": erro_absoluto_medio(alvo_fixo, previsto),
                   "segundos": time.perf_counter() - t0})

tabela_sementes = pd.DataFrame(linhas).set_index("semente").round(3)
display(tabela_sementes)
print(f"a mesma origem, o mesmo código: MAE de {tabela_sementes['MAE'].min():.3f} a "
      f"{tabela_sementes['MAE'].max():.3f}, diferença de "
      f"{tabela_sementes['MAE'].max() - tabela_sementes['MAE'].min():.3f}")


A mesma origem, o mesmo código, o mesmo número de épocas: o MAE vai de 2,861 a 3,219, uma diferença de 0,358.
Para dar escala, a diferença entre o ETS e o SARIMAX no backtesting é de 0,448. Ou seja: **a variação entre
duas sementes chega a oito décimos da diferença entre dois métodos clássicos**.

Isso muda a decisão prática: uma rede só pode ser comparada com outra se a nota vier com a faixa das
sementes, do mesmo jeito que uma previsão só significa algo com o backtesting e não com uma janela só.

## O modelo fundacional: previsão sem treino

O modelo fundacional de série foi treinado em milhares de séries de domínios diferentes e aprendeu um
repertório de formas: tendência, ciclos, choque, platô. Na sua série ele não treina nada: você entrega os
últimos valores como contexto e ele devolve a previsão, junto com quantis.

Três consequências práticas:

- **não há hiperparâmetro para ajustar** e não há treino, então ele é o método mais rápido de colocar de pé,
  e o mais rápido de rodar (a previsão custa menos de um décimo de segundo);
- **ele devolve intervalo**, o que o ETS e o SARIMAX também dão, mas o modelo de árvore e a GRU não;
- **há um cuidado científico**: um modelo pré-treinado em muitas séries públicas pode ter visto algo parecido
  com a sua durante o treino. Com série sintética como esta, a comparação mede capacidade, não pureza. No
  seu dado real, vale desconfiar de resultado bom demais.

O `Chronos` da Amazon é uma dessas famílias, e a variante `bolt` é a mais leve. A célula carrega o modelo e
roda o mesmo backtesting.


In [ ]:
%pip install chronos-forecasting==2.3.2

from chronos import ChronosBoltPipeline

t0 = time.perf_counter()
pipeline = ChronosBoltPipeline.from_pretrained("amazon/chronos-bolt-small", device_map="cpu")
tempo_carga = time.perf_counter() - t0
print(f"carregar o modelo: {tempo_carga:.1f} s | "
      f"parâmetros: {sum(p.numel() for p in pipeline.model.parameters()) / 1e6:.1f} M")


def previsor_chronos(treino, horizonte):
    """Sem treino: o contexto entra e a mediana sai."""
    contexto = torch.tensor(treino.iloc[-JANELA:].to_numpy(dtype=np.float32))
    quantis, _ = pipeline.predict_quantiles(contexto, prediction_length=horizonte,
                                            quantile_levels=[0.1, 0.5, 0.9])
    return quantis[0, :, 1].numpy()


t0 = time.perf_counter()
_, passos_chronos = backtest(serie, previsor_chronos)
tempo_chronos = time.perf_counter() - t0

print(f"Chronos no backtesting:")
print(f"  MAE médio {passos_chronos.mean():.3f} | passo 1 {passos_chronos[:, 0].mean():.3f} | "
      f"passo 14 {passos_chronos[:, -1].mean():.3f}")
print(f"  tempo: {tempo_chronos:.1f} s em {passos_chronos.shape[0]} origens "
      f"({tempo_chronos / passos_chronos.shape[0]:.2f} s por previsão, sem treino)")

contexto = torch.tensor(serie.iloc[-JANELA:].to_numpy(dtype=np.float32))
quantis, _ = pipeline.predict_quantiles(contexto, prediction_length=14,
                                        quantile_levels=[0.1, 0.5, 0.9])
print()
print(f"largura média do intervalo 10-90%: {float((quantis[0, :, 2] - quantis[0, :, 0]).mean()):.2f}")


O fundacional zerou o trabalho de treino e ficou com 3,686 de MAE médio: à frente do sazonal ingênuo (4,040)
e **da rede que precisou de treino** (4,329), e atrás do ETS (2,947) e do SARIMAX (3,395). No passo 1 ele é
o melhor de todos, com 2,474.

O custo é de outra natureza: menos de 1 segundo para carregar o modelo e cerca de 0,03 segundo por previsão,
contra segundos de treino por origem da GRU. E ele devolveu o intervalo de 10% a 90% com largura média de
20,04, sem custo adicional e sem nenhum hiperparâmetro a escolher.

O cuidado que fica registrado: um modelo pré-treinado em milhares de séries públicas pode ter visto algo
parecido com a sua durante o treino. Numa série sintética como esta, o número mede capacidade, não pureza.
No seu dado real, vale desconfiar de resultado bom demais, e a checagem é sempre a mesma: backtesting com
origens que o modelo não viu.

## O confronto final, com os dois extremos dentro

Todos os candidatos na mesma janela de avaliação, no mesmo horizonte.


In [ ]:
_, passos_ingenuo = backtest(serie, lambda t, h: np.resize(t.iloc[-PERIODO:].to_numpy(), h))
_, passos_ets = backtest(serie, lambda t, h: ExponentialSmoothing(
    t, trend="add", seasonal="add", seasonal_periods=PERIODO).fit().forecast(h).to_numpy())
_, passos_sarimax = backtest(serie, lambda t, h: SARIMAX(
    t, order=(1, 1, 1), seasonal_order=(1, 1, 0, PERIODO),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False).get_forecast(h).predicted_mean.to_numpy())

candidatos = {
    "sazonal ingênuo": passos_ingenuo,
    "ETS(A,A,A)": passos_ets,
    "SARIMAX(1,1,1)(1,1,0,7)": passos_sarimax,
    "GRU (janela de 28)": passos_gru,
    "Chronos bolt small (zero-shot)": passos_chronos,
}

comparacao = pd.DataFrame({
    "MAE médio": [p.mean() for p in candidatos.values()],
    "MAE no passo 1": [p[:, 0].mean() for p in candidatos.values()],
    "MAE no passo 14": [p[:, -1].mean() for p in candidatos.values()],
}, index=list(candidatos)).round(3)
display(comparacao)

comparacao[["MAE no passo 1", "MAE no passo 14"]].plot(kind="bar")
plt.title("os cinco candidatos, no primeiro e no último passo do horizonte")
plt.ylabel("MAE")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


A tabela fecha a pasta, e a ordem dela é desconfortável para quem espera que o mais novo vença:

1. **ETS(A,A,A), quatro linhas de código**: 2,947 de MAE médio, o melhor;
2. **SARIMAX(1,1,1)(1,1,0,7)**: 3,395, e o melhor no passo 14, com 2,713;
3. **Chronos zero-shot, sem treinar nada**: 3,686;
4. **sazonal ingênuo, o sarrafo**: 4,040;
5. **GRU treinada por pouco mais de 20 segundos**: 4,329.

As colunas de passo mostram a divisão de águas: no passo 1 os dois modelos de sequência são os melhores
(2,474 do fundacional e 2,632 da GRU) e no passo 14 são os piores (5,961 e 7,237), enquanto o SARIMAX faz
2,713 e o próprio ingênuo faz 3,791. Prever um passo é uma tarefa; prever catorze em sequência é outra.

Nada disso diz que rede neural não serve. Diz que, quando a série tem a estrutura que o ETS supõe, com
tendência e semana estáveis, quatro linhas ganham de um modelo de 47 milhões de parâmetros. E diz que o
fundacional, que não treina, não escolhe hiperparâmetro e ainda entrega intervalo, é o melhor custo-benefício
entre os métodos modernos.

## Resumo

Os dois extremos da prática moderna, medidos na mesma série dos outros três notebooks.

A GRU recebeu a janela de 28 valores, sem nenhuma informação de calendário, precisou ser escalonada, validada
no fim do treino e parada no ponto certo, e ainda variou entre sementes numa faixa de 0,358 de MAE, oito
décimos da diferença entre o ETS e o SARIMAX. No backtesting ela fez 4,329 de MAE médio, ficando atrás do
sarrafo no conjunto, ainda que tenha sido a melhor no passo 1 entre os modelos treinados.

O modelo fundacional não treinou nada: recebeu o mesmo contexto e devolveu mediana e quantis em cerca de 0,03
segundo por previsão, com 3,686 de MAE médio. Ficou à frente do sarrafo e da rede, e atrás dos dois clássicos.

A conclusão da pasta inteira fica completa com este notebook. O protocolo (divisão temporal, baselines,
horizonte, backtesting, métrica escolhida pelo problema) é o que permite comparar ETS, SARIMAX, boosting, rede
neural e modelo fundacional na mesma régua, e é ele que revela o que cada um custa: o ETS custa quatro linhas
e zero segundos, o fundacional custa uma chamada, o boosting custa uma tabela de atributos, e a rede custa
treino, semente e hiperparâmetro.

**O que levar para o seu dado:** comece pelo sazonal ingênuo, meça ETS e SARIMAX, e só depois gaste tempo com
o resto. Se a série tem atributo externo (feriado, promoção, clima), o boosting entra, e é o único dos cinco
que aceita essa informação. Se você tem muitas séries e pouco tempo, o fundacional zero-shot é o melhor
custo-benefício entre os modernos. Rede neural se paga quando há dado de sobra e padrão que os outros não
pegam, e aí ela precisa vir com sementes, validação no fim do treino e escala calculada no treino.

## Referências

```bibtex
@article{ansari2024chronos,
  title   = {Chronos: Learning the Language of Time Series},
  author  = {Ansari, Abdul Fatir and Stella, Lorenzo and Turkmen, Caner and Zhang, Xiyuan and Mercado, Pedro and Shen, Huibin and others},
  journal = {Transactions on Machine Learning Research},
  year    = {2024},
  url     = {https://arxiv.org/abs/2403.07815}
}

@inproceedings{woo2024moirai,
  title     = {Unified Training of Universal Time Series Forecasting Transformers},
  author    = {Woo, Gerald and Liu, Chenghao and Kumar, Akshat and Xiong, Caiming and Savarese, Silvio and Sahoo, Doyen},
  booktitle = {International Conference on Machine Learning},
  year      = {2024},
  url       = {https://arxiv.org/abs/2402.02592}
}

@inproceedings{das2024timesfm,
  title     = {A Decoder-Only Foundation Model for Time-Series Forecasting},
  author    = {Das, Abhimanyu and Kong, Weihao and Sen, Rajat and Zhou, Yichen},
  booktitle = {International Conference on Machine Learning},
  year      = {2024},
  url       = {https://arxiv.org/abs/2310.10688}
}

@book{hyndman2021forecasting,
  title     = {Forecasting: Principles and Practice},
  author    = {Hyndman, Rob J. and Athanasopoulos, George},
  edition   = {3},
  publisher = {OTexts},
  year      = {2021},
  url       = {https://otexts.com/fpp3/}
}
```
